In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [6]:


url = "https://lagosministryofhealth.org/primary-health-facilities/"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)

print(response.status_code)

200


In [ ]:


tables = pd.read_html(response.text)

print(f"Number of tables found: {len(tables)}")



In [12]:
print(type(response.text))
print(response.text[:500])

<class 'str'>
<!doctype html>
<html dir="ltr" lang="en-US" prefix="og: https://ogp.me/ns#">
<head>
<meta charset="UTF-8">    
<meta name="viewport" content="width=device-width, initial-scale=1">
<meta name="format-detection" content="telephone=no">

<title>Primary Health Facilities - Ministry of health</title>

		<!-- All in One SEO 4.9.10 - aioseo.com -->
	<meta name="description" content="PRIMARY HEALTH CARE (PHC) Lagos State Primary Health Facilities – Healthcare at the Community Level Primary health care 


In [13]:
from io import StringIO
import pandas as pd

tables = pd.read_html(StringIO(response.text))

print(f"Number of tables found: {len(tables)}")

Number of tables found: 6


In [14]:
for i, table in enumerate(tables):
    print(f"\n===== TABLE {i+1} =====")
    print(table.head())
    print(table.columns)
    print(table.shape)


===== TABLE 1 =====
   S/N             LGA/LCDA             WARD     NAME OF PHC  \
0    1         ALIMOSHO LGA          ALABATA    AKOWONJO PHC   
1    2                  NaN  OGUNTADE BAMEKE  ORISUNBARE PHC   
2    3  AGBADO OKE ODO LCDA    AGBADO OKEODO     OKE ODO PHC   
3    4                  NaN   AJASA/AMIKANLE   SURULERE PHC.   
4    5                  NaN              NaN    AMIKANLE PHC   

                                       ADDRESS  
0             100 AKOWONJO ROAD,VULGANIDER B/S  
1          20,EJIGBO ROAD,ORISUNMIBARE, SHASHA  
2         EKORO OKEODO ROAD, OKEODO ABULE EGBA  
3  3, FAITH RESCUE BY COMMAND SECONDARY SCHOOL  
4                      2/5, FADIPE STR, AGBADO  
Index(['S/N', 'LGA/LCDA', 'WARD', 'NAME OF PHC', 'ADDRESS'], dtype='str')
(66, 5)

===== TABLE 2 =====
   S/N              LGA/LCDA       WARD        NAME OF H/F  \
0    1  IGBOGBO BAIYEKU LCDA   ITUMOKUN        IGBOGBO PHC   
1    2                   NaN     BAYEKU         BAYEKU PHC   
2    3     

In [15]:
df = pd.concat(tables, ignore_index=True)

print(df.shape)
df.head()

(325, 7)


,S/N,LGA/LCDA,WARD,NAME OF PHC,ADDRESS,NAME OF H/F,NAME OF FACILITY
0,1.0,ALIMOSHO LGA,ALABATA,AKOWONJO PHC,"100 AKOWONJO ROAD,VULGANIDER B/S",NaN,NaN
1,2.0,NaN,OGUNTADE BAMEKE,ORISUNBARE PHC,"20,EJIGBO ROAD,ORISUNMIBARE, SHASHA",NaN,NaN
2,3.0,AGBADO OKE ODO LCDA,AGBADO OKEODO,OKE ODO PHC,"EKORO OKEODO ROAD, OKEODO ABULE EGBA",NaN,NaN
3,4.0,NaN,AJASA/AMIKANLE,SURULERE PHC.,"3, FAITH RESCUE BY COMMAND SECONDARY SCHOOL",NaN,NaN
4,5.0,NaN,NaN,AMIKANLE PHC,"2/5, FADIPE STR, AGBADO",NaN,NaN


In [16]:
all_tables = []

for i, table in enumerate(tables, start=1):
    table["District"] = f"District {i}"
    all_tables.append(table)

df = pd.concat(all_tables, ignore_index=True)

In [35]:
df.isnull().sum()

S/N         60
LGA/LCDA     0
WARD         0
ADDRESS      1
District     0
PHC_NAME     0
dtype: int64

In [17]:
df.head()

,S/N,LGA/LCDA,WARD,NAME OF PHC,ADDRESS,District,NAME OF H/F,NAME OF FACILITY
0,1.0,ALIMOSHO LGA,ALABATA,AKOWONJO PHC,"100 AKOWONJO ROAD,VULGANIDER B/S",District 1,NaN,NaN
1,2.0,NaN,OGUNTADE BAMEKE,ORISUNBARE PHC,"20,EJIGBO ROAD,ORISUNMIBARE, SHASHA",District 1,NaN,NaN
2,3.0,AGBADO OKE ODO LCDA,AGBADO OKEODO,OKE ODO PHC,"EKORO OKEODO ROAD, OKEODO ABULE EGBA",District 1,NaN,NaN
3,4.0,NaN,AJASA/AMIKANLE,SURULERE PHC.,"3, FAITH RESCUE BY COMMAND SECONDARY SCHOOL",District 1,NaN,NaN
4,5.0,NaN,NaN,AMIKANLE PHC,"2/5, FADIPE STR, AGBADO",District 1,NaN,NaN


In [19]:
cols_to_fill = ["LGA/LCDA", "WARD"]

df[cols_to_fill] = df[cols_to_fill].ffill()

In [21]:
df.head()

,S/N,LGA/LCDA,WARD,NAME OF PHC,ADDRESS,District,NAME OF H/F,NAME OF FACILITY
0,1.0,ALIMOSHO LGA,ALABATA,AKOWONJO PHC,"100 AKOWONJO ROAD,VULGANIDER B/S",District 1,NaN,NaN
1,2.0,ALIMOSHO LGA,OGUNTADE BAMEKE,ORISUNBARE PHC,"20,EJIGBO ROAD,ORISUNMIBARE, SHASHA",District 1,NaN,NaN
2,3.0,AGBADO OKE ODO LCDA,AGBADO OKEODO,OKE ODO PHC,"EKORO OKEODO ROAD, OKEODO ABULE EGBA",District 1,NaN,NaN
3,4.0,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,SURULERE PHC.,"3, FAITH RESCUE BY COMMAND SECONDARY SCHOOL",District 1,NaN,NaN
4,5.0,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,AMIKANLE PHC,"2/5, FADIPE STR, AGBADO",District 1,NaN,NaN


In [22]:
df[df["NAME OF PHC"].astype(str).str.strip() == ""]

,S/N,LGA/LCDA,WARD,NAME OF PHC,ADDRESS,District,NAME OF H/F,NAME OF FACILITY


In [23]:
missing_phc = df[
    df["NAME OF PHC"].isna() |
    (df["NAME OF PHC"].astype(str).str.strip() == "")
]

missing_phc

,S/N,LGA/LCDA,WARD,NAME OF PHC,ADDRESS,District,NAME OF H/F,NAME OF FACILITY
66,1.0,IGBOGBO BAIYEKU LCDA,ITUMOKUN,NaN,"AGBELE ROAD,IGBOGBO",District 2,IGBOGBO PHC,NaN
67,2.0,IGBOGBO BAIYEKU LCDA,BAYEKU,NaN,"BAYEKU ROAD,IGBOGBO",District 2,BAYEKU PHC,NaN
68,3.0,IGBOGBO BAIYEKU LCDA,BAYEKU,NaN,OPP CEDARFIED NUR/PRI BAYEKU IKORODU,District 2,LAJO H/POST,NaN
69,4.0,IGBOGBO BAIYEKU LCDA,BAYEKU,NaN,"OFFIN LAST BUS STOP, OFFIN",District 2,OFFIN HEALTH POST,NaN
70,5.0,IGBOGBO BAIYEKU LCDA,EWU ELEPE,NaN,"ELEPE B/STOP,ELEPE",District 2,EWU ELEPE PHC,NaN
...,...,...,...,...,...,...,...,...
320,41.0,ISOLO LCDA,AKINBAYE,NaN,"189, MUSHIN ROAD, ISOLO",District 6,ISOLO PHC,NaN
321,42.0,ISOLO LCDA,OKOTA,NaN,"14, AYANWUN STREET, FREEDOM ESTATE OKOTA",District 6,OKOTA PHC,NaN
322,43.0,ISOLO LCDA,ALAGBEJI,NaN,"ROTARY CLUB BUILDING, ILE IWE META BUS STOP, I...",District 6,LSDPC PHC,NaN
323,44.0,ISOLO LCDA,APENA,NaN,"BAMGBALA STREET, ALLI DADA B/STOP",District 6,APENA PHC,NaN


In [26]:
df.columns



Index(['S/N', 'LGA/LCDA', 'WARD', 'NAME OF PHC', 'ADDRESS', 'District',
       'NAME OF H/F', 'NAME OF FACILITY'],
      dtype='str')

In [27]:
df[["NAME OF PHC", "NAME OF H/F", "NAME OF FACILITY"]].notna().sum()

NAME OF PHC          66
NAME OF H/F         194
NAME OF FACILITY     65
dtype: int64

In [28]:
df[["District", "NAME OF PHC", "NAME OF H/F", "NAME OF FACILITY"]].head(30)

,District,NAME OF PHC,NAME OF H/F,NAME OF FACILITY
0,District 1,AKOWONJO PHC,NaN,NaN
1,District 1,ORISUNBARE PHC,NaN,NaN
2,District 1,OKE ODO PHC,NaN,NaN
3,District 1,SURULERE PHC.,NaN,NaN
4,District 1,AMIKANLE PHC,NaN,NaN
5,District 1,IKOLA PHC,NaN,NaN
6,District 1,OLOTA PHC,NaN,NaN
7,District 1,ABORU PHC,NaN,NaN
8,District 1,TINUBU PHC,NaN,NaN
9,District 1,OGUNDIMU PHC,NaN,NaN


In [ ]:
df.groupby("District")[["NAME OF PHC", "NAME OF H/F", "NAME OF FACILITY"]].count()

In [30]:
df["PHC_NAME"] = (
    df["NAME OF PHC"]
    .fillna(df["NAME OF H/F"])
    .fillna(df["NAME OF FACILITY"])
)

In [31]:
df["PHC_NAME"].isna().sum()

np.int64(0)

In [32]:
df = df.drop(columns=[
    "NAME OF PHC",
    "NAME OF H/F",
    "NAME OF FACILITY"
])

In [33]:
df.head()

,S/N,LGA/LCDA,WARD,ADDRESS,District,PHC_NAME
0,1.0,ALIMOSHO LGA,ALABATA,"100 AKOWONJO ROAD,VULGANIDER B/S",District 1,AKOWONJO PHC
1,2.0,ALIMOSHO LGA,OGUNTADE BAMEKE,"20,EJIGBO ROAD,ORISUNMIBARE, SHASHA",District 1,ORISUNBARE PHC
2,3.0,AGBADO OKE ODO LCDA,AGBADO OKEODO,"EKORO OKEODO ROAD, OKEODO ABULE EGBA",District 1,OKE ODO PHC
3,4.0,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,"3, FAITH RESCUE BY COMMAND SECONDARY SCHOOL",District 1,SURULERE PHC.
4,5.0,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,"2/5, FADIPE STR, AGBADO",District 1,AMIKANLE PHC


In [36]:
df[df["ADDRESS"].isna()]

,S/N,LGA/LCDA,WARD,ADDRESS,District,PHC_NAME
72,7.0,IGBOGBO BAIYEKU LCDA,IBESHE,NaN,District 2,MOJODA PHC


In [37]:
len(df)

325

In [38]:
df["S/N"] = range(1, len(df) + 1)

In [39]:
df.head()

,S/N,LGA/LCDA,WARD,ADDRESS,District,PHC_NAME
0,1,ALIMOSHO LGA,ALABATA,"100 AKOWONJO ROAD,VULGANIDER B/S",District 1,AKOWONJO PHC
1,2,ALIMOSHO LGA,OGUNTADE BAMEKE,"20,EJIGBO ROAD,ORISUNMIBARE, SHASHA",District 1,ORISUNBARE PHC
2,3,AGBADO OKE ODO LCDA,AGBADO OKEODO,"EKORO OKEODO ROAD, OKEODO ABULE EGBA",District 1,OKE ODO PHC
3,4,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,"3, FAITH RESCUE BY COMMAND SECONDARY SCHOOL",District 1,SURULERE PHC.
4,5,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,"2/5, FADIPE STR, AGBADO",District 1,AMIKANLE PHC


In [40]:
df.isnull().sum()

S/N         0
LGA/LCDA    0
WARD        0
ADDRESS     1
District    0
PHC_NAME    0
dtype: int64

In [41]:
df.head()

,S/N,LGA/LCDA,WARD,ADDRESS,District,PHC_NAME
0,1,ALIMOSHO LGA,ALABATA,"100 AKOWONJO ROAD,VULGANIDER B/S",District 1,AKOWONJO PHC
1,2,ALIMOSHO LGA,OGUNTADE BAMEKE,"20,EJIGBO ROAD,ORISUNMIBARE, SHASHA",District 1,ORISUNBARE PHC
2,3,AGBADO OKE ODO LCDA,AGBADO OKEODO,"EKORO OKEODO ROAD, OKEODO ABULE EGBA",District 1,OKE ODO PHC
3,4,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,"3, FAITH RESCUE BY COMMAND SECONDARY SCHOOL",District 1,SURULERE PHC.
4,5,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,"2/5, FADIPE STR, AGBADO",District 1,AMIKANLE PHC


In [42]:
df.duplicated().sum()


np.int64(0)

In [43]:
import os

output_path = os.path.join("..",
    "data",
    "processed",
    "lagos_primary_health_centres_cleaned.xlsx"
)

df.to_excel(output_path, index=False)

In [44]:
csv_path = os.path.join("..",
    "data",
    "processed",
    "lagos_primary_health_centres_cleaned.csv"
)

df.to_csv(csv_path, index=False)

In [45]:


print(os.path.exists(output_path))
print(os.path.exists(csv_path))

True
True


In [ ]:
from dotenv import load_dotenv
load_dotenv()
gmaps_key=googlemaps.Client(key=os.getenv("GOOGLE_API_KEY"))
df['lat']=None
df['lon']=None

for i in range(len(df)):
    geocode_result=gmaps_key.geocode(df.loc[i,'ADDRESS'])
    try:
        lat=geocode_result[0]["geometry"]["location"]["lat"]
        lng=geocode_result[0]["geometry"]["location"]["lng"]
        df.loc[i,'lat']=lat
        df.loc[i,'lon']=lng
    except:
        lat=None
        lng=None
print(df)        